# Deploy [mistralai/Voxtral-4B-TTS-2603](https://huggingface.co/mistralai/Voxtral-4B-TTS-2603) on SageMaker AI

This notebook demonstrates how to deploy **Voxtral-4B-TTS-2603** a frontier, open-weights text-to-speech model using vLLM-Omni on Amazon SageMaker AI with bidirectional streaming.


## Voxtral TTS Key Features

- Realistic, expressive speech with natural prosody across **9 languages** (EN, FR, ES, DE, IT, PT, NL, AR, HI)
- **20 preset voices** with easy adaptation to new voices
- **24 kHz** audio output in WAV, PCM, FLAC, MP3, AAC, and Opus formats
- Very low latency with fast time-to-first-audio
- BF16 weights, runs on a single GPU with >= 16GB memory
- Production-ready performance for high-throughput, real-time voice agent workflows

## What We'll Do

1. **Setup Environment**: Install dependencies and configure AWS
2. **Build Custom Container**: Create vLLM-Omni container with TTS support
3. **Deploy to SageMaker**: Create endpoint for inference
4. **Test TTS**: Validate text-to-speech via bidirectional streaming

## Prerequisites

- AWS credentials configured
- Docker environment available
- HuggingFace access to `[mistralai/Voxtral-4B-TTS-2603](https://huggingface.co/mistralai/Voxtral-4B-TTS-2603)`

## 1. Environment Setup

In [ ]:
%pip install -r requirements.txt

In [ ]:
import boto3
import os
from rich import print as rprint
from rich.pretty import pprint
from sagemaker.core.helper.session_helper import Session

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix
s3_client = boto3.client("s3")

## 2. Install Docker

In [ ]:
%%bash

# see https://docs.docker.com/engine/install/ubuntu/#install-using-the-repository
sudo apt-get update
sudo apt-get install -y ca-certificates curl
sudo install -m 0755 -d /etc/apt/keyrings
sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc
sudo chmod a+r /etc/apt/keyrings/docker.asc

# Add the repository to Apt sources:
echo \
  "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu \
  $(. /etc/os-release && echo \"$VERSION_CODENAME\") stable" | \
  sudo tee /etc/apt/sources.list.d/docker.list > /dev/null
sudo apt-get update

## Currently only Docker version 20.10.X is supported in Studio
# pick the latest patch from:
# apt-cache madison docker-ce | awk '{ print $3 }' | grep -i 20.10
VERSION_STRING=5:20.10.24~3-0~ubuntu-jammy
sudo apt-get install docker-ce-cli=$VERSION_STRING docker-compose-plugin -y

# validate the Docker Client is able to access Docker Server at [unix:///docker/proxy.sock]
docker version

## 3. Create Docker Artifacts for vLLM-Omni TTS Deployment

In [ ]:
!mkdir -p voxtral-tts-4B-docker-artifacts

print("Docker artifacts directory created")

In [ ]:
%%writefile voxtral-tts-4B-docker-artifacts/Dockerfile
FROM public.ecr.aws/deep-learning-containers/vllm:omni-sagemaker-cuda-v1-soci

# Set a docker label to advertise bidirectional streaming support on the container
LABEL com.amazonaws.sagemaker.capabilities.bidirectional-streaming=true

# Set working directory
WORKDIR /opt/ml/code

# Install additional TTS dependencies (vllm-omni is already in the base image)
COPY requirements.txt .
RUN pip install --upgrade --no-cache-dir -r requirements.txt

# Application code bridge
COPY app.py .

COPY sagemaker-entrypoint.sh entrypoint.sh
RUN chmod +x entrypoint.sh

ENTRYPOINT ["./entrypoint.sh"]

HEALTHCHECK --interval=30s --timeout=10s --start-period=120s --retries=3 \
  CMD curl -f http://localhost:$PORT/ping || exit 1

In [ ]:
%%writefile voxtral-tts-4B-docker-artifacts/sagemaker-entrypoint.sh
#!/bin/bash
# Starts vLLM-Omni on :8081 (internal) and the TTS bridge on :8080 (SageMaker-facing).
# The bridge routes:
#   /invocations                       -> /v1/audio/speech
#   /invocations-bidirectional-stream  -> /v1/audio/speech (WebSocket)

PREFIX="SM_VLLM_"
ARG_PREFIX="--"

# Bridge configuration
VLLM_INTERNAL_PORT="${VLLM_INTERNAL_PORT:-8081}"
BRIDGE_PORT=8080
BRIDGE_SCRIPT=/opt/ml/code/app.py

# Build vLLM args from SM_VLLM_ environment variables
ARGS=(--port "${VLLM_INTERNAL_PORT}" --omni)

# Loop through all environment variables
while IFS='=' read -r key value; do
    arg_name=$(echo "${key#"${PREFIX}"}" | tr '[:upper:]' '[:lower:]' | tr '_' '-')
    ARGS+=("${ARG_PREFIX}${arg_name}")
    [[ -n "$value" ]] && ARGS+=("$value")
done < <(env | grep "^${PREFIX}")


echo "============================================"
echo "  vLLM-Omni TTS args: ${ARGS[*]}"
echo "  vLLM internal port: ${VLLM_INTERNAL_PORT}"
echo "  Bridge port: ${BRIDGE_PORT}"
echo "============================================"

# Signal handling
cleanup() {
    echo "Shutting down..."
    kill $VLLM_PID $BRIDGE_PID 2>/dev/null
    wait
}
trap cleanup SIGTERM SIGINT

# 1. Start vLLM-Omni on internal port (background)
vllm-omni serve "${ARGS[@]}" &
VLLM_PID=$!

# 2. Wait for vLLM to be ready
echo "Waiting for vLLM-Omni on port ${VLLM_INTERNAL_PORT}..."
MAX_RETRIES=120
RETRY_COUNT=0
until curl -sf "http://localhost:${VLLM_INTERNAL_PORT}/health" > /dev/null 2>&1; do
    RETRY_COUNT=$((RETRY_COUNT + 1))
    if [ $RETRY_COUNT -ge $MAX_RETRIES ]; then
        echo "ERROR: vLLM failed to start after ${MAX_RETRIES} retries"
        kill $VLLM_PID 2>/dev/null
        exit 1
    fi
    # Check if vLLM process is still alive
    if ! kill -0 $VLLM_PID 2>/dev/null; then
        echo "ERROR: vLLM process died"
        exit 1
    fi
    sleep 2
done
echo "vLLM-Omni ready on port ${VLLM_INTERNAL_PORT}"

# 3. Start the TTS bridge on port 8080 (SageMaker-facing)
echo "Starting TTS bridge on port ${BRIDGE_PORT}..."
export VLLM_INTERNAL_PORT
export BRIDGE_PORT
python3 "${BRIDGE_SCRIPT}" &
BRIDGE_PID=$!

echo "Bridge ready on port ${BRIDGE_PORT}"

# 4. Wait for either process to exit -- if one dies, shut down both
wait -n $VLLM_PID $BRIDGE_PID
EXIT_CODE=$?

cleanup
exit $EXIT_CODE

In [ ]:
%%writefile voxtral-tts-4B-docker-artifacts/build_and_push.sh
#!/bin/bash

# Build and push Docker image to ECR
REPO_NAME=$1
VERSION_TAG=$2
REGION=$3
ACCOUNT_ID=$4

# Create ECR repository if it doesn't exist
aws ecr describe-repositories --repository-names $REPO_NAME --region $REGION || \
aws ecr create-repository --repository-name $REPO_NAME --region $REGION

# Get ECR login token
aws ecr get-login-password --region $REGION | docker login --username AWS --password-stdin $ACCOUNT_ID.dkr.ecr.$REGION.amazonaws.com

# Build Docker image
docker build --network sagemaker -t $REPO_NAME:$VERSION_TAG .

# Tag image for ECR
docker tag $REPO_NAME:$VERSION_TAG $ACCOUNT_ID.dkr.ecr.$REGION.amazonaws.com/$REPO_NAME:$VERSION_TAG

# Push to ECR
docker push $ACCOUNT_ID.dkr.ecr.$REGION.amazonaws.com/$REPO_NAME:$VERSION_TAG

echo "Image pushed to: $ACCOUNT_ID.dkr.ecr.$REGION.amazonaws.com/$REPO_NAME:$VERSION_TAG"

In [ ]:
# Make scripts executable
!chmod +x voxtral-tts-4B-docker-artifacts/sagemaker-entrypoint.sh
!chmod +x voxtral-tts-4B-docker-artifacts/build_and_push.sh

print("Docker scripts created and made executable")

## 4. Build and Push Docker Image

In [ ]:
account_id = "<>" # Your AWS account ID
region = sess.boto_region_name

In [ ]:
# Configuration for Docker image
REPO_NAME = "voxtral-4b-tts-2603-vllm-omni"
VERSION_TAG = "latest"

print(f"Building Docker image: {REPO_NAME}:{VERSION_TAG}")
print(f"Target ECR: {account_id}.dkr.ecr.{region}.amazonaws.com/{REPO_NAME}:{VERSION_TAG}")

In [ ]:
%%bash -s {REPO_NAME} {VERSION_TAG} {region} {account_id}

REPO_NAME=$1
VERSION_TAG=$2
REGION=$3
ACCOUNT_ID=$4

echo "Building and pushing Docker image..."
cd voxtral-tts-4B-docker-artifacts && bash build_and_push.sh $REPO_NAME $VERSION_TAG $REGION $ACCOUNT_ID

In [ ]:
# Get the image URI
inference_image = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{REPO_NAME}:{VERSION_TAG}"
print(f"Docker image URI: {inference_image}")

## 5. Download the Model from Hugging Face and Upload to Amazon S3

**Best Practices:**
> **Store Models in Your Own S3 Bucket** — For production use-cases, always download and store model files in your own S3 bucket to ensure validated artifacts.

In [ ]:
HF_MODEL_ID = "mistralai/Voxtral-4B-TTS-2603"

base_name = HF_MODEL_ID.split('/')[-1].replace('.', '-').lower()
model_lineage = HF_MODEL_ID.split("/")[0]
base_name

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import os
import sagemaker
import jinja2

# - This will download the model into the current directory where ever the jupyter notebook is running
local_model_path = Path(".")
local_model_path.mkdir(exist_ok=True)
model_name = HF_MODEL_ID
# - Leverage the snapshot library to donload the model since the model is stored in repository using LFS
model_download_path = snapshot_download(
    repo_id=model_name,
    cache_dir=local_model_path,
    ignore_patterns=["README.md", ".gitattributes"],
)

### Upload model files to S3

SageMaker AI allows us to provide [uncompressed](https://docs.aws.amazon.com/sagemaker/latest/dg/large-model-inference-uncompressed.html) files.

> **Note**: The default SageMaker bucket follows the naming pattern: `sagemaker-{region}-{account-id}`

In [ ]:
s3_model_prefix = "hf-large-models/Voxtral_4B_TTS_2603"

In [ ]:
model_artifact = sess.upload_data(path=model_download_path, key_prefix=s3_model_prefix)
print(f"Model uploaded to --- > {model_artifact}")

## 6. Configure Model Environment

Environment variables prefixed with `SM_VLLM_` are automatically converted to vLLM CLI arguments by the entrypoint script (e.g., `SM_VLLM_MODEL` → `--model`, `SM_VLLM_TOKENIZER_MODE` → `--tokenizer-mode`).

In [ ]:
instance_count = 1
instance_type = "ml.g6.4xlarge"  # 1x NVIDIA L4 (24GB) — sufficient for BF16 TTS model
number_of_gpu = 1
health_check_timeout = 700

In [ ]:
# Environment variables for vLLM-Omni TTS
vllm_env = {
    # Model configuration
    "SM_VLLM_MODEL": "/opt/ml/model",  # SageMaker downloads model here
    "SM_VLLM_TOKENIZER_MODE": "mistral",  # Required for Voxtral TTS tokenizer
    "HF_TOKEN": os.environ.get("HF_TOKEN", ""),
    # Performance configuration
    "SM_VLLM_MAX_MODEL_LEN": "4096",
    "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.9",
    "SM_VLLM_KV_CACHE_DTYPE": "auto",
    "SM_VLLM_MAX_NUM_SEQS": "4",
    "SM_VLLM_DTYPE": "auto",
    "SM_VLLM_TENSOR_PARALLEL_SIZE": str(number_of_gpu),
}

## 7. Deploy Model to SageMaker Endpoint

In [ ]:
import time

model_name = f"Voxtral-4B-TTS-2603-{int(time.time())}"
endpoint_config_name = f"Voxtral-4B-TTS-2603-config-{int(time.time())}"
endpoint_name = f"Voxtral-4B-TTS-2603-endpoint-{int(time.time())}"

print(f"Model name: {model_name}")
print(f"Endpoint name: {endpoint_name}")

In [ ]:
from sagemaker.core.resources import Model
from sagemaker.core.shapes import (
    ContainerDefinition,
    ModelDataSource,
    S3ModelDataSource,
)
from sagemaker.core.helper.session_helper import get_execution_role

role = get_execution_role()

voxtral_tts_model = Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(
        image=inference_image,
        model_data_source=ModelDataSource(
            s3_data_source=S3ModelDataSource(
                s3_uri=f"{model_artifact}/",
                s3_data_type="S3Prefix",
                compression_type="None",
            )
        ),
        environment=vllm_env,
    ),
    execution_role_arn=role,
)

pprint(voxtral_tts_model)

### Create Endpoint Configuration

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig
from sagemaker.core.shapes import ProductionVariant

print(f"Creating EndpointConfig: {endpoint_config_name}")
endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=model_name,
            initial_variant_weight=1.0,
            instance_type=instance_type,
            initial_instance_count=1,
            model_data_download_timeout_in_seconds=health_check_timeout,
        )
    ],
)

### Create Endpoint

In [ ]:
print(f"Creating Endpoint: {endpoint_name}")
endpoint = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_config_name,
)
endpoint.wait_for_status("InService")
print(f"Endpoint {endpoint_name} is InService")

## 8. Test the Deployed TTS Model

### Available Voices

Voxtral TTS provides 20 preset voices:

| Voice | Description |
|-------|-------------|
| `casual_male` | Casual male voice |
| `casual_female` | Casual female voice |
| `cheerful_female` | Cheerful female voice |
| ... | See model documentation for full list |

### Test 1: TTS via Bidirectional Streaming Client

Uses the bidirectional streaming transport to send text and receive audio.

In [ ]:
!python sagemaker_tts_bidi_client.py \
    {endpoint_name} \
    "Hello! This is a test of the Voxtral text-to-speech model deployed on Amazon SageMaker AI." \
    --model "/opt/ml/model" \
    --voice casual_female \
    --output tts_output.wav

In [ ]:
# Play the generated audio
from IPython.display import Audio, display
import soundfile as sf

audio_data, sample_rate = sf.read("tts_output.wav")
print(f"Generated audio: {len(audio_data)} samples at {sample_rate} Hz ({len(audio_data)/sample_rate:.1f}s)")
display(Audio(audio_data, rate=sample_rate))

### Test 2: TTS in Multiple Languages

Voxtral TTS supports 9 languages: English, French, Spanish, German, Italian, Portuguese, Dutch, Arabic, and Hindi.

In [ ]:
# Test French
!python sagemaker_tts_bidi_client.py \
    {endpoint_name} \
    "Bonjour! Paris est une ville magnifique avec une riche histoire culturelle." \
    --model "/opt/ml/model" \
    --voice fr_male \
    --output tts_output_fr.wav

In [ ]:
audio_data, sample_rate = sf.read("tts_output_fr.wav")
print(f"French audio: {len(audio_data)/sample_rate:.1f}s")
display(Audio(audio_data, rate=sample_rate))

## 9. Cleanup (Optional)

In [ ]:
# print("Deleting endpoint to avoid charges...")
# endpoint.delete()
# endpoint_config.delete()
# voxtral_tts_model.delete()

print("To delete the endpoint later, uncomment the code above or use:")
print(f"aws sagemaker delete-endpoint --endpoint-name {endpoint_name}")